In [14]:
import pandas as pd
import folium

# Load dataset
df = pd.read_csv("../data/raw/crime_dataset_india.csv")

# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Normalize city names
df["city"] = df["city"].astype(str).str.strip().str.lower()

print("Total rows:", len(df))
print("Unique cities:", df["city"].unique()[:30])


Total rows: 40160
Unique cities: ['ahmedabad' 'chennai' 'ludhiana' 'pune' 'delhi' 'mumbai' 'surat'
 'visakhapatnam' 'bangalore' 'kolkata' 'ghaziabad' 'hyderabad' 'jaipur'
 'lucknow' 'bhopal' 'patna' 'kanpur' 'varanasi' 'nagpur' 'meerut' 'thane'
 'indore' 'rajkot' 'vasai' 'agra' 'kalyan' 'nashik' 'srinagar' 'faridabad']


In [15]:
df["city"] = (
    df["city"]
    .astype(str)
    .str.strip()
    .str.lower()
)


In [16]:
df.columns

Index(['report_number', 'date_reported', 'date_of_occurrence',
       'time_of_occurrence', 'city', 'crime_code', 'crime_description',
       'victim_age', 'victim_gender', 'weapon_used', 'crime_domain',
       'police_deployed', 'case_closed', 'date_case_closed'],
      dtype='object')

In [17]:
from city_coordinates import CITY_COORDINATES
# Map lat/lon using CITY_COORDINATES
# fallback to center of India if city not found
df["latitude"] = df["city"].map(lambda x: CITY_COORDINATES.get(x, (22.5937, 78.9629))[0])
df["longitude"] = df["city"].map(lambda x: CITY_COORDINATES.get(x, (22.5937, 78.9629))[1])

print("Sample coordinates:")
print(df[["city", "latitude", "longitude"]].head())
df = df.dropna(subset=["latitude", "longitude"])
print("Total rows after mapping:", len(df))
df[["city", "latitude", "longitude"]].head()


Sample coordinates:
        city  latitude  longitude
0  ahmedabad   23.0225    72.5714
1    chennai   13.0827    80.2707
2   ludhiana   30.9010    75.8573
3       pune   18.5204    73.8567
4       pune   18.5204    73.8567
Total rows after mapping: 40160


,city,latitude,longitude
0,ahmedabad,23.0225,72.5714
1,chennai,13.0827,80.2707
2,ludhiana,30.9010,75.8573
3,pune,18.5204,73.8567
4,pune,18.5204,73.8567


# CREATE BASE MAP

In [18]:
import folium
import os

# create map
india_map = folium.Map(location=[22.5937, 78.9629], zoom_start=5)

# add markers for all cities
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color="red",
        fill=True,
        fill_opacity=0.7,
        popup=row["city"].title()
    ).add_to(india_map)

# create outputs folder safely
output_dir = os.path.join(os.getcwd(), "outputs")
os.makedirs(output_dir, exist_ok=True)

# save map HTML
html_file = os.path.join(output_dir, "crime_hotspots_india.html")
india_map.save(html_file)

print(f"Map saved successfully: {html_file}")


Map saved successfully: C:\Users\Tanish_Gupta\OneDrive\Desktop\ML Projects\ai-crime-hotspot-prediction\notebooks\outputs\crime_hotspots_india.html
